# nb02 — Train ReliefForge (Kaggle, GPU T4 x2 → use cuda:0)

**Settings → Accelerator: GPU T4 x2, Internet: on.** Add the dataset from nb01 as Input
(`/kaggle/input/reliefforge-dataset`). Add your HF write token as the Secret `HF_TOKEN`
if you want checkpoint pushes (optional).

Plan: 200-step smoke test → measure steps/s → set the step budget → Stage A → Stage B.
Everything is logged to MLflow (`/kaggle/working/mlruns`) — download the folder with the
notebook output and copy it into the repo root (course evidence).

In [ ]:
!nvidia-smi && nproc && free -g | head -2

In [ ]:
%%bash
cd /kaggle/working
rm -rf reliefforge
git clone --depth 1 https://github.com/<YOUR_USERNAME>/reliefforge.git reliefforge
pip -q install PyYAML mlflow
ls /kaggle/input/reliefforge-dataset | head

In [ ]:
%%bash
cd /kaggle/working/reliefforge
# 200-step smoke: measures steps/s and proves the wiring on this GPU
python src/train.py --config configs/stage_a.yaml --data /kaggle/input/reliefforge-dataset \
  --smoke --out /kaggle/working/smoke_a 2>&1 | tail -5

In [ ]:
# Set the budget from the measured images/s (aim <= 8-10 GPU-hours total, PRD_v2 §7.5)
# T4 reference: batch 8 at 518 is usually ~2.5-4 img/s forward+backward.
STAGE_A_STEPS = 2000
STAGE_B_STEPS = 8000

In [ ]:
%%bash -s "$STAGE_A_STEPS"
cd /kaggle/working/reliefforge
python src/train.py --config configs/stage_a.yaml --data /kaggle/input/reliefforge-dataset \
  --out /kaggle/working/train_a $([ "$1" != "2000" ] && echo "" || echo "") 2>&1 | tail -8
# if you changed the step counts, edit configs/stage_a.yaml / stage_b.yaml in the repo first

In [ ]:
%%bash
cd /kaggle/working/reliefforge
python src/train.py --config configs/stage_b.yaml --data /kaggle/input/reliefforge-dataset \
  --stage-a-dir /kaggle/working/train_a/export --out /kaggle/working/train_b 2>&1 | tail -8

In [ ]:
%%bash
cd /kaggle/working/reliefforge
# quick check on the val split before the full evaluation in nb03
python src/evaluate.py --config configs/config.yaml --data /kaggle/input/reliefforge-dataset \
  --model /kaggle/working/train_b/export --label stage_b --split val --max-items 60 2>&1 | tail -5
python src/evaluate.py --config configs/config.yaml --data /kaggle/input/reliefforge-dataset \
  --zero-shot --split val --max-items 60 2>&1 | tail -5

In [ ]:
!du -sh /kaggle/working/train_a/export /kaggle/working/train_b/export /kaggle/working/mlruns
!tar -czf /kaggle/working/reliefforge_outputs.tar.gz -C /kaggle/working train_a train_b mlruns
!ls -lh /kaggle/working/reliefforge_outputs.tar.gz
# After Save & Run All: download reliefforge_outputs.tar.gz,
# unpack, copy mlruns/ into the repo root, commit; keep the exports for nb03 / the app.